In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [ ]:
# 1. Konfigurasi
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3

os.makedirs("weights", exist_ok=True)

In [ ]:
# 2. Augmentasi & Preprocessing
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# 3. Data Loader
train_data = datasets.ImageFolder("dataset/train", transform=train_transform)
val_data = datasets.ImageFolder("dataset/val", transform=val_transform)

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

print(f"[+] Device: {DEVICE}")
print(f"[+] Kelas: {train_data.classes}")

In [ ]:
# 4. Model Architecture (MobileNetV3-Small)
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
# 5. Training Loop
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)

In [ ]:
    # Evaluasi Validasi
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data).item()
            total_val += labels.size(0)

    train_acc = correct_train / total_train
    val_acc = correct_val / total_val

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {running_loss/total_train:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "weights/best_model.pth")
        print(f" -> Model terbaik disimpan (Val Acc: {best_val_acc:.4f})")

print("\n[✓] Training selesai!")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

DEVICE = torch.device("cpu")
NUM_CLASSES = 7

model = models.mobilenet_v3_small()
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, NUM_CLASSES)

model.load_state_dict(
    torch.load("weights/best_model.pth", map_location=DEVICE)
)
model.eval()

# Dummy input statis: 1 batch, 3 channel (RGB), 224x224
dummy_input = torch.randn(1, 3, 224, 224, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    "weights/ai1_model.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes=None,  # Fixed shape untuk latensi minimal di CPU
)

print("[✓] Model berhasil diekspor ke weights/ai1_model.onnx")